# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [222]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nicol\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Q1

In [223]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
# TODO: apply sent_tokenize

#Same content that i created for the S03 practice, is generall, but it protects it part of the text including the acronyms.
sentence_pattern_protected = r'(?<=[.!?])\s+(?=[A-Z])'
sentences = re.split(sentence_pattern_protected, text)

# The print is able to show the new sentences created 3 at first, but the one created with nltk is able to show 4 senteces and 
# It will show UPC and UNESCO in two different sentences, but i dont know how to fix it.
# I could delete the dots that creates the acronyms, but based on what its needed, i dont know if it is the best option.
print("New sentence")
for s in sentences:
    print("-", s)


#Then, i follow a quick MLjorney to easily split the sentences using the nltk requested. It works well.
sentences = nltk.sent_tokenize(text)
print(sentences)


New sentence
- In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.
- He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O.
- A report valued the project at $3.2 billion.
['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C.', 'and U.N.E.S.C.O.', 'A report valued the project at $3.2 billion.']


## Q2

In [224]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
# Return: text_norm
# print(text_norm)

text_norm = text

#For acronyms this is a simple regex that removes the dots searchng from A to Z capitals and follow by dots to remove them
text_norm = re.sub(r'([A-Z])\.', r'\1', text_norm)

# For meters we use a lambda function to convert the meters to centimeters, i know it is used very commonly as i see in StackOverflow
text_norm = re.sub(r'(\d+\.\d+)m', 
    lambda m: f"{int(float(m.group(1))*100)} centimeters", text_norm)

# As it asks in the question i wil create a easy dicctionary for the numbers

numbers = {
    '0': 'zero',
    '1': 'one',
    '2': 'two',
    '3': 'three',
    '4': 'four',
    '5': 'five',
    '6': 'six',
    '7': 'seven',
    '8': 'eight',
    '9': 'nine'
}

#Then, with them i create a simple function similar to the beforre class excersices we have donde where i use the group to get the numbers and then i use the dictionary to get the words
# Again, its simple, but it works using the same logic as before.

def money_translator(match): 
    d1 = numbers[match.group(1)] 
    d2 = numbers[match.group(2)]
    return f"{d1} point {d2} billion"

text_norm = re.sub(r'\$(\d)\.(\d) billion', money_translator, text_norm)


print(text_norm)





In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q3

In [225]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case
text_case = text


# Here im just telling the model that if it finds a space between Sam and Altman (only this two words in this secuences) it should join them with an underscore.
text_case = re.sub(r'(Sam)\s(Altman)', r'\1_\2', text_case)

# This is a "bucle" that will go through each word in the text and check if it is a proper noun or not.
# It just go from word 0 to word n and check if the first letter is uppercase or if there is an uppercase letter in the word.
words_position = []
for word in text_case.split():
    if word[0].isupper() or any(c.isupper() for c in word[1:]):
        words_position.append(word)
    else:
        words_position.append(word.lower())
        
text_case = " ".join(words_position)

# print(text_case)
print(text_case)

In mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q4

In [226]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

# If im not getting it wrong, the tokenization will be done with just the same nltk that we  used before, then a print.
tokens = tokens = nltk.word_tokenize(text_case)

# print(tokens)
print(tokens)


['In', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '1.86m', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'U.P.C', '.', 'and', 'U.N.E.S.C.O', '.', 'A', 'report', 'valued', 'the', 'project', 'at', '$', '3.2', 'billion', '.']


## Q5

In [227]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop

# Just created a stopwords that works in english to full an empty list with all the non stop words
stopwords = nltk.corpus.stopwords.words('english')
tokens_nostop = []

# I did a easy "bucle" jsut to tell the model that i want to study all words with no differents in capuital or lower case
for i in tokens:
     if i.lower() not in stopwords and i not in [',', '.', '$']:
        tokens_nostop.append(i)

# At first i did the lsit without the  []',', '.', '$'] where i saw that the model was using them


# print(tokens_nostop)
print(tokens_nostop)




['mid-February', '2026', 'CEO', 'OpenAI', 'Sam_Altman', 'visited', 'Barcelona', '1.86m', 'tall', 'met', 'researchers', 'U.P.C', 'U.N.E.S.C.O', 'report', 'valued', 'project', '3.2', 'billion']


## Q6

In [228]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]
bigrams = [] 
for i in range(len(tokens_nostop) - 1):
    pair = (tokens_nostop[i], tokens_nostop[i+1])
    bigrams.append(pair)
    
# print(bigrams)
print(bigrams[:5])


[('mid-February', '2026'), ('2026', 'CEO'), ('CEO', 'OpenAI'), ('OpenAI', 'Sam_Altman'), ('Sam_Altman', 'visited')]


## Q7

In [229]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

# At first i thouth i wolud not need the model, so i commented it, as i did the exxecrise i realised i needed
bigram_counts = Counter(bigrams)
context_counts = Counter(tokens_nostop)
model = defaultdict(dict)


for (w1, w2), count in bigram_counts.items():
    prob = count/context_counts[w1]
    model[w1][w2] = prob


 
def predict_next(prev_word, model, top_k=3):
    if prev_word in model:        
        opt = model[prev_word].items()        
        sorted_opt = sorted(opt, key=lambda x: x[1], reverse=True)
        return sorted_opt[:top_k]
    
    return []
    
test = tokens_nostop[0] 

    
print(predict_next("CEO",model, top_k=3))
# Example:  
# print(predict_next("Barcelona", model, top_k=3))


[('OpenAI', 1.0)]


## Q8

In [ ]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

words = corpus.split()

# Here i tried to get the vocab with the words separetad in spaces // This command is extracted from MLjoruney
vocab = {" ".join(list(word)) + " </w>": 1 for word in words}


# The function get the vocab and them create the pairs, in this case it is adapted but also from MLjourney
def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs


def merge_vocab(pair, v_in):
    v_out = {}
    s_pair = ' '.join(pair)
    replacement = ''.join(pair)
    for word in v_in:
        w_out = word.replace(s_pair, replacement)
        v_out[w_out] = v_in[word]
    return v_out


merges = []
num_merges = 5
print("Merges list:", merges)

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)

# I ran out of time, so i jsut created a non error code that solves part of the execrsie, due to the lack of time and resources
# i cannot complete it.


Merges list: []


## Q9

In [231]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = 75
FP = 10
FN = 5
TN = 10

# Formules created bases on what we did today in class. 
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * (precision * recall) / (precision + recall)

# print(accuracy, precision, recall, f1)
# Same as with the formules, print same as the one we saw in class
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Accuracy: 0.85
Precision: 0.8823529411764706
Recall: 0.9375
F1 Score: 0.9090909090909091
